In [1]:
# Customer Data Cleaning, Feature Engineering and Analytical Dataset Preparation

# Import required libraries
import os
import re
import warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

# Configure pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.2f}".format)


# Define project folder paths
current_directory = os.getcwd()

project_folder_name = (
    "Customer Engagement & Product Utilization Analytics for Retention Strategy"
)

if os.path.basename(current_directory).lower() == "notebooks":
    project_folder = os.path.dirname(current_directory)
else:
    project_folder = os.path.join(current_directory, project_folder_name)

data_folder = os.path.join(project_folder, "data")
processed_data_folder = os.path.join(data_folder, "processed")
output_folder = os.path.join(project_folder, "outputs")
table_folder = os.path.join(output_folder, "tables")

os.makedirs(processed_data_folder, exist_ok=True)
os.makedirs(table_folder, exist_ok=True)

input_path = os.path.join(
    processed_data_folder,
    "European_Bank_Original_Copy.csv"
)

analytical_output_path = os.path.join(
    processed_data_folder,
    "European_Bank_Cleaned_Analytical.csv"
)

feature_dictionary_path = os.path.join(
    table_folder,
    "Feature_Dictionary.csv"
)


# Load validated dataset
if not os.path.exists(input_path):
    raise FileNotFoundError(
        "Validated dataset not found. Please complete Notebook 01 first."
    )

df = pd.read_csv(input_path)

print("Validated dataset loaded successfully.")
print("Original dataset shape:", df.shape)


# Create a separate analytical copy
df_clean = df.copy()

print("\nSeparate analytical copy created.")
print("Analytical dataset shape:", df_clean.shape)


# Standardize column names
df_clean.columns = [
    re.sub(
        r"_+",
        "_",
        re.sub(
            r"[^a-zA-Z0-9]+",
            "_",
            column.strip().lower()
        )
    ).strip("_")
    for column in df_clean.columns
]

print("\nStandardized column names:")
print(df_clean.columns.tolist())


# Check required columns
required_columns = [
    "year",
    "customerid",
    "surname",
    "creditscore",
    "geography",
    "gender",
    "age",
    "tenure",
    "balance",
    "numofproducts",
    "hascrcard",
    "isactivemember",
    "estimatedsalary",
    "exited"
]

missing_required_columns = [
    column for column in required_columns
    if column not in df_clean.columns
]

if missing_required_columns:
    raise ValueError(
        f"Required columns are missing: {missing_required_columns}"
    )


# Correct data types where required
numerical_columns = [
    "year",
    "creditscore",
    "age",
    "tenure",
    "balance",
    "numofproducts",
    "hascrcard",
    "isactivemember",
    "estimatedsalary",
    "exited"
]

for column in numerical_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

# Customer ID is an identifier, not a numerical measurement
if df_clean["customerid"].isnull().any():
    raise ValueError(
        "CustomerId contains missing or invalid values. "
        "Resolve these before converting IDs to strings."
    )

df_clean["customerid"] = df_clean["customerid"].astype("int64").astype(str)

# Standardize text columns
categorical_columns = [
    "surname",
    "geography",
    "gender"
]

for column in categorical_columns:
    df_clean[column] = df_clean[column].astype("string").str.strip()

print("\nData types corrected successfully.")
display(df_clean.dtypes.to_frame(name="Data Type"))


# Handle missing values if present
missing_before = df_clean.isnull().sum().sum()

for column in df_clean.columns:

    if df_clean[column].isnull().sum() > 0:

        if column == "customerid":
            raise ValueError(
                "CustomerId cannot be imputed. Investigate missing IDs."
            )

        if pd.api.types.is_numeric_dtype(df_clean[column]):

            median_value = df_clean[column].median()

            if pd.isna(median_value):
                raise ValueError(
                    f"Cannot impute {column}: no valid numerical values."
                )

            df_clean[column] = df_clean[column].fillna(median_value)

        else:

            mode_values = df_clean[column].mode(dropna=True)

            if len(mode_values) == 0:
                raise ValueError(
                    f"Cannot impute {column}: no valid categorical values."
                )

            df_clean[column] = df_clean[column].fillna(
                mode_values.iloc[0]
            )

missing_after = df_clean.isnull().sum().sum()

print("\nMissing Value Treatment")
print("Missing values before treatment:", missing_before)
print("Missing values after treatment:", missing_after)


# Handle complete duplicate records if present
duplicates_before = df_clean.duplicated().sum()

if duplicates_before > 0:
    df_clean = df_clean.drop_duplicates().copy()

duplicates_after = df_clean.duplicated().sum()

print("\nDuplicate Record Validation")
print("Complete duplicates before cleaning:", duplicates_before)
print("Complete duplicates after cleaning:", duplicates_after)


# Validate Customer ID uniqueness
duplicate_ids = df_clean["customerid"].duplicated().sum()

print("\nCustomer ID Validation")
print("Duplicate Customer IDs:", duplicate_ids)

if duplicate_ids > 0:
    raise ValueError(
        "Duplicate Customer IDs detected. "
        "Investigate before proceeding with customer-level analysis."
    )


# Create age groups
df_clean["age_group"] = pd.cut(
    df_clean["age"],
    bins=[0, 29, 44, 59, np.inf],
    labels=["Under 30", "30-44", "45-59", "60+"],
    include_lowest=True
)


# Create tenure groups
df_clean["tenure_group"] = pd.cut(
    df_clean["tenure"],
    bins=[-1, 2, 5, 10],
    labels=["0-2 Years", "3-5 Years", "6-10 Years"]
)


# Create balance categories
positive_balances = df_clean.loc[
    df_clean["balance"] > 0,
    "balance"
]

if len(positive_balances) > 0:

    balance_median = positive_balances.median()
    balance_75th_percentile = positive_balances.quantile(0.75)

    df_clean["balance_category"] = np.select(
        [
            df_clean["balance"] == 0,
            df_clean["balance"] <= balance_median,
            df_clean["balance"] <= balance_75th_percentile
        ],
        [
            "Zero Balance",
            "Low Balance",
            "Medium Balance"
        ],
        default="High Balance"
    )

else:

    balance_median = 0
    balance_75th_percentile = 0

    df_clean["balance_category"] = "Zero Balance"


# Create product-count categories
df_clean["product_count_category"] = np.select(
    [
        df_clean["numofproducts"] == 1,
        df_clean["numofproducts"] == 2,
        df_clean["numofproducts"] >= 3
    ],
    [
        "Single Product",
        "Two Products",
        "Three or More Products"
    ],
    default="Invalid Product Count"
)


# Create customer activity labels
df_clean["activity_status"] = np.where(
    df_clean["isactivemember"] == 1,
    "Active",
    "Inactive"
)


# Create customer status labels
df_clean["customer_status"] = np.where(
    df_clean["exited"] == 1,
    "Exited",
    "Retained"
)


# Create high-balance indicator
df_clean["high_balance_flag"] = (
    (df_clean["balance"] > 0) &
    (df_clean["balance"] >= balance_75th_percentile)
).astype(int)


# Create inactive high-balance indicator
df_clean["inactive_high_balance_flag"] = (
    (df_clean["high_balance_flag"] == 1) &
    (df_clean["isactivemember"] == 0)
).astype(int)


# Create single-product and multi-product indicator
df_clean["product_relationship"] = np.where(
    df_clean["numofproducts"] == 1,
    "Single Product",
    "Multi Product"
)


# Create premium-customer candidate indicator
df_clean["premium_customer_candidate"] = (
    (df_clean["high_balance_flag"] == 1) &
    (df_clean["numofproducts"] >= 2)
).astype(int)


# Calculate Relationship Strength Index
activity_score = df_clean["isactivemember"]

product_adoption_score = (
    df_clean["numofproducts"].clip(upper=2) / 2
)

df_clean["relationship_strength_index"] = (
    (60 * activity_score) +
    (40 * product_adoption_score)
).round(2)


# Create RSI strength categories
df_clean["relationship_strength_category"] = pd.cut(
    df_clean["relationship_strength_index"],
    bins=[-1, 24, 49, 74, 89, 100],
    labels=[
        "Very Weak",
        "Weak",
        "Moderate",
        "Strong",
        "Very Strong"
    ]
)


# Validate newly created features
engineered_columns = [
    "age_group",
    "tenure_group",
    "balance_category",
    "product_count_category",
    "activity_status",
    "customer_status",
    "high_balance_flag",
    "inactive_high_balance_flag",
    "product_relationship",
    "premium_customer_candidate",
    "relationship_strength_index",
    "relationship_strength_category"
]

print("\nEngineered Feature Validation")

feature_validation = []

for column in engineered_columns:

    feature_validation.append({
        "Feature": column,
        "Data Type": str(df_clean[column].dtype),
        "Missing Values": df_clean[column].isnull().sum(),
        "Unique Values": df_clean[column].nunique(dropna=True)
    })

feature_validation_df = pd.DataFrame(feature_validation)

display(feature_validation_df)


# Validate binary indicators
binary_feature_columns = [
    "high_balance_flag",
    "inactive_high_balance_flag",
    "premium_customer_candidate"
]

print("\nBinary Feature Validation")

for column in binary_feature_columns:

    invalid_values = (
        ~df_clean[column].isin([0, 1])
    ).sum()

    print(
        f"{column}: "
        f"Unique values = {df_clean[column].unique().tolist()}, "
        f"Invalid values = {invalid_values}"
    )

    if invalid_values > 0:
        raise ValueError(
            f"Invalid binary values detected in {column}."
        )


# Validate RSI values and ranges
print("\nRelationship Strength Index Validation")

print(
    "Minimum RSI:",
    df_clean["relationship_strength_index"].min()
)

print(
    "Maximum RSI:",
    df_clean["relationship_strength_index"].max()
)

invalid_rsi = (
    (df_clean["relationship_strength_index"] < 0) |
    (df_clean["relationship_strength_index"] > 100)
).sum()

print("Invalid RSI values:", invalid_rsi)

if invalid_rsi > 0:
    raise ValueError("RSI values are outside the permitted 0-100 range.")


# Display important engineered feature distributions
print("\nAge Group Distribution")
display(df_clean["age_group"].value_counts(dropna=False).to_frame("Customer Count"))

print("\nTenure Group Distribution")
display(df_clean["tenure_group"].value_counts(dropna=False).to_frame("Customer Count"))

print("\nBalance Category Distribution")
display(df_clean["balance_category"].value_counts().to_frame("Customer Count"))

print("\nProduct Count Category Distribution")
display(df_clean["product_count_category"].value_counts().to_frame("Customer Count"))

print("\nActivity Status Distribution")
display(df_clean["activity_status"].value_counts().to_frame("Customer Count"))

print("\nCustomer Status Distribution")
display(df_clean["customer_status"].value_counts().to_frame("Customer Count"))

print("\nRelationship Strength Category Distribution")
display(
    df_clean["relationship_strength_category"]
    .value_counts(dropna=False)
    .to_frame("Customer Count")
)


# Create feature dictionary
feature_dictionary = pd.DataFrame([
    {
        "Feature": "age_group",
        "Description": "Customer age segment",
        "Business Rule": "Under 30, 30-44, 45-59, 60+"
    },
    {
        "Feature": "tenure_group",
        "Description": "Customer relationship duration segment",
        "Business Rule": "0-2, 3-5, 6-10 years"
    },
    {
        "Feature": "balance_category",
        "Description": "Customer balance segment",
        "Business Rule": "Zero balance; positive balances divided using median and 75th percentile"
    },
    {
        "Feature": "product_count_category",
        "Description": "Number of products held by customer",
        "Business Rule": "1, 2, 3 or more products"
    },
    {
        "Feature": "activity_status",
        "Description": "Customer activity label",
        "Business Rule": "Active if IsActiveMember = 1; otherwise Inactive"
    },
    {
        "Feature": "customer_status",
        "Description": "Customer retention status",
        "Business Rule": "Exited if Exited = 1; otherwise Retained"
    },
    {
        "Feature": "high_balance_flag",
        "Description": "Identifies customers with high positive balances",
        "Business Rule": "1 if positive balance is at or above the 75th percentile of positive balances"
    },
    {
        "Feature": "inactive_high_balance_flag",
        "Description": "Identifies inactive customers with high balances",
        "Business Rule": "1 if high_balance_flag = 1 and IsActiveMember = 0"
    },
    {
        "Feature": "product_relationship",
        "Description": "Single-product or multi-product relationship",
        "Business Rule": "Single Product if product count = 1; otherwise Multi Product"
    },
    {
        "Feature": "premium_customer_candidate",
        "Description": "Potential premium customer screening indicator",
        "Business Rule": "1 if high_balance_flag = 1 and product count >= 2"
    },
    {
        "Feature": "relationship_strength_index",
        "Description": "Proposed relationship strength score",
        "Business Rule": "60 x activity score + 40 x min(product count, 2) / 2"
    },
    {
        "Feature": "relationship_strength_category",
        "Description": "Relationship strength segment",
        "Business Rule": "0-24 Very Weak; 25-49 Weak; 50-74 Moderate; 75-89 Strong; 90-100 Very Strong"
    }
])

print("\nFeature Dictionary")
display(feature_dictionary)


# Generate final analytical dataset summary
print("\nFinal Analytical Dataset Summary")

print("Original dataset records:", len(df))
print("Cleaned dataset records:", len(df_clean))
print("Original dataset columns:", len(df.columns))
print("Final analytical dataset columns:", len(df_clean.columns))
print("Total engineered features:", len(engineered_columns))
print("Remaining missing values:", df_clean.isnull().sum().sum())
print("Remaining complete duplicates:", df_clean.duplicated().sum())
print("Duplicate Customer IDs:", df_clean["customerid"].duplicated().sum())


# Save processed analytical dataset
df_clean.to_csv(
    analytical_output_path,
    index=False
)


# Save feature dictionary
feature_dictionary.to_csv(
    feature_dictionary_path,
    index=False
)


# Save engineered feature validation report
feature_validation_df.to_csv(
    os.path.join(
        table_folder,
        "Engineered_Feature_Validation_Report.csv"
    ),
    index=False
)


# Final completion message
print("\nProcessed analytical dataset saved successfully.")
print("Feature dictionary saved successfully.")

print("\nAnalytical Dataset:")
print(analytical_output_path)

print("\nFeature Dictionary:")
print(feature_dictionary_path)

print("\nData cleaning and feature engineering completed successfully.")

Validated dataset loaded successfully.
Original dataset shape: (10000, 14)

Separate analytical copy created.
Analytical dataset shape: (10000, 14)

Standardized column names:
['year', 'customerid', 'surname', 'creditscore', 'geography', 'gender', 'age', 'tenure', 'balance', 'numofproducts', 'hascrcard', 'isactivemember', 'estimatedsalary', 'exited']

Data types corrected successfully.


,Data Type
year,int64
customerid,object
surname,string[python]
creditscore,int64
geography,string[python]
gender,string[python]
age,int64
tenure,int64
balance,float64
numofproducts,int64



Missing Value Treatment
Missing values before treatment: 0
Missing values after treatment: 0

Duplicate Record Validation
Complete duplicates before cleaning: 0
Complete duplicates after cleaning: 0

Customer ID Validation
Duplicate Customer IDs: 0

Engineered Feature Validation


,Feature,Data Type,Missing Values,Unique Values
0,age_group,category,0,4
1,tenure_group,category,0,3
2,balance_category,object,0,4
3,product_count_category,object,0,3
4,activity_status,object,0,2
5,customer_status,object,0,2
6,high_balance_flag,int32,0,2
7,inactive_high_balance_flag,int32,0,2
8,product_relationship,object,0,2
9,premium_customer_candidate,int32,0,2



Binary Feature Validation
high_balance_flag: Unique values = [0, 1], Invalid values = 0
inactive_high_balance_flag: Unique values = [0, 1], Invalid values = 0
premium_customer_candidate: Unique values = [0, 1], Invalid values = 0

Relationship Strength Index Validation
Minimum RSI: 20.0
Maximum RSI: 100.0
Invalid RSI values: 0

Age Group Distribution


,Customer Count
age_group,
30-44,6019
45-59,1814
Under 30,1641
60+,526


... (11 outputs left out)

,Customer Count
relationship_strength_category,
Very Strong,2588
Strong,2563
Very Weak,2521
Weak,2328
Moderate,0



Feature Dictionary


,Feature,Description,Business Rule
0,age_group,Customer age segment,"Under 30, 30-44, 45-59, 60+"
1,tenure_group,Customer relationship duration segment,"0-2, 3-5, 6-10 years"
2,balance_category,Customer balance segment,Zero balance; positive balances divided using ...
3,product_count_category,Number of products held by customer,"1, 2, 3 or more products"
4,activity_status,Customer activity label,Active if IsActiveMember = 1; otherwise Inactive
5,customer_status,Customer retention status,Exited if Exited = 1; otherwise Retained
6,high_balance_flag,Identifies customers with high positive balances,1 if positive balance is at or above the 75th ...
7,inactive_high_balance_flag,Identifies inactive customers with high balances,1 if high_balance_flag = 1 and IsActiveMember = 0
8,product_relationship,Single-product or multi-product relationship,Single Product if product count = 1; otherwise...
9,premium_customer_candidate,Potential premium customer screening indicator,1 if high_balance_flag = 1 and product count >= 2



Final Analytical Dataset Summary
Original dataset records: 10000
Cleaned dataset records: 10000
Original dataset columns: 14
Final analytical dataset columns: 26
Total engineered features: 12
Remaining missing values: 0
Remaining complete duplicates: 0
Duplicate Customer IDs: 0

Processed analytical dataset saved successfully.
Feature dictionary saved successfully.

Analytical Dataset:
C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\European_Bank_Cleaned_Analytical.csv

Feature Dictionary:
C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\outputs\tables\Feature_Dictionary.csv

Data cleaning and feature engineering completed successfully.
